# Daily Sales and Revenue Forecasting
Chronological forecasting using the real Olist PostgreSQL database.

## 1. Business Problem
Forecast near-term daily orders and revenue for staffing, fulfillment, and cash planning without treating point forecasts as guarantees.

In [1]:
from pathlib import Path
import sys,json,pandas as pd
PROJECT_ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(PROJECT_ROOT))
from src.models.forecasting.build_series import build_daily_series
from src.models.forecasting.train import main as train_forecasts
from src.models.forecasting.forecast import generate_forecast

INFO: generated new fontManager


## 2. Time-Series Construction
Payments are aggregated to order grain before daily revenue. Canceled and unavailable orders are excluded, and missing calendar days are explicitly filled with zero.

In [2]:
full,modeling=build_daily_series()
print('Full:',full.date.min(),full.date.max(),len(full),'revenue',full.daily_revenue.sum())
print('Modeling:',modeling.index.min(),modeling.index.max(),len(modeling))

Full: 2016-09-04 00:00:00 2018-09-03 00:00:00 730 revenue 15739137.01
Modeling: 2017-01-01 00:00:00 2018-08-21 00:00:00 598


## 3. Data Quality and Coverage
The sparse 2016 launch period and incomplete September 2018 tail remain in reconciliation diagnostics but are excluded from model fitting and evaluation.

In [3]:
display(modeling.describe())
print('Duplicate dates:',modeling.index.duplicated().sum(),'missing days:',modeling.isna().sum().sum())

,daily_orders,daily_revenue
count,598.000000,598.000000
mean,162.586957,26085.422391
std,90.321230,14500.785275
min,0.000000,0.000000
25%,103.000000,15732.297500
50%,150.000000,24041.720000
75%,217.750000,34445.185000
max,1166.000000,178450.110000


Duplicate dates: 0 missing days: 0


## 4. Time-Series EDA
Daily histories, volatility, 7-day and 30-day rolling levels, spikes, and trend changes are retained and visualized.

## 5. Seasonality
Weekday averages and monthly aggregates evaluate weekly operational patterns and changing marketplace scale.

## 6. Baselines
Last value, historical mean, same weekday last week, and seven-day rolling mean establish meaningful non-ML benchmarks.

## 7. Feature Engineering
Lags 1–28, shifted rolling statistics, calendar fields, weekend status, and a time index are computed from history only. Recursive forecasts use previous predictions when actual future targets are unavailable.

## 8. Backtesting Strategy
Three expanding-window 30-day folds compare every model on identical future periods. Chronological validation is required because random splitting would leak future market conditions.

## 9. Model Comparison
Linear regression, random forest, and histogram gradient boosting are compared with baselines using average MAE and RMSE. ML is not forced to win.

In [4]:
train_forecasts()

INFO: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


INFO: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


INFO: Selected orders=RandomForestRegressor revenue=HistGradientBoostingRegressor


      date  predicted_orders  predicted_revenue  orders_lower_bound  orders_upper_bound  revenue_lower_bound  revenue_upper_bound
2018-08-22        233.697828       37675.033751          121.501293          329.421150         20283.258147         53921.845902
2018-08-23        231.902267       32983.990424          112.899870          333.432182         14537.226725         50216.336991
2018-08-24        204.072883       34229.965687           78.633343          311.094810         14785.369437         52394.453880
2018-08-25        182.148933       26822.821381           50.586834          294.394478          6429.156785         45873.897321
2018-08-26        187.072751       26794.114798           49.660620          304.309399          5493.626823         46692.314656
2018-08-27        247.672415       39361.322520          104.649335          369.696187         17191.071725         60072.025568
2018-08-28        221.665600       34262.391489           73.243535          348.295652   

In [5]:
report_dir=PROJECT_ROOT/'reports'/'forecasting'
comparison=pd.read_csv(report_dir/'model_comparison.csv')
display(comparison.groupby(['target','model'])[['mae','rmse','mape','smape']].mean())

mae          rmse  \
target        model                                                       
daily_orders  HistGradientBoostingRegressor     48.648151     60.894255   
              HistoricalMean                    67.096502     80.117345   
              LinearRegression                  78.826797     90.659215   
              NaiveLastValue                    56.688889     66.904771   
              RandomForestRegressor             40.697236     52.244299   
              RollingMean7                      48.783481     59.000351   
              SeasonalNaive7                    44.122222     55.885657   
daily_revenue HistGradientBoostingRegressor   8024.505629   9825.604448   
              HistoricalMean                 11879.468399  14326.679832   
              LinearRegression               12826.440129  14914.333329   
              NaiveLastValue                 10015.217333  11914.046927   
              RandomForestRegressor           8482.869798  10405.043860   
              RollingMean7                    8459.167703  10563.752631   
              SeasonalNaive7                  8424.681222  10241.383538   

                                                  mape      smape  
target        model                                                
daily_orders  HistGradientBoostingRegressor  25.455358  25.269154  
              HistoricalMean                 29.393687  34.314451  
              LinearRegression               47.049640  34.522179  
              NaiveLastValue                 29.293612  28.736950  
              RandomForestRegressor          23.722242  20.330880  
              RollingMean7                   26.792456  24.291097  
              SeasonalNaive7                 24.882265  22.222507  
daily_revenue HistGradientBoostingRegressor  27.168799  23.481902  
              HistoricalMean                 31.398960  37.029402  
              LinearRegression               46.590817  33.948430  
              NaiveLastValue                 31.163548  31.606586  
              RandomForestRegressor          27.146371  25.569424  
              RollingMean7                   27.274901  25.375120  
              SeasonalNaive7                 27.897785  25.934839

## 10. Final Holdout Evaluation
The last 30 modeling days remain untouched until model selection. Metadata reports 7-, 14-, and 30-day prefix performance.

In [6]:
metadata=json.loads((PROJECT_ROOT/'models'/'forecasting'/'forecast_metadata.json').read_text())
display(metadata['holdout_metrics'])

{'daily_orders': {'7': {'mae': 12.178035497835525,
   'rmse': 16.626391029862763,
   'mape': 5.9060154225114845,
   'smape': 5.622627954369748},
  '14': {'mae': 37.334747052947044,
   'rmse': 49.4147939687642,
   'mape': 13.675580889213926,
   'smape': 15.086318106281897},
  '30': {'mae': 43.02610836533835,
   'rmse': 55.702832854112145,
   'mape': 14.879759830265368,
   'smape': 16.63261577923284}},
 'daily_revenue': {'7': {'mae': 4087.5116864494526,
   'rmse': 4661.65761550917,
   'mape': 10.645932421710524,
   'smape': 11.286873472962233},
  '14': {'mae': 7453.309467233465,
   'rmse': 8826.354229362105,
   'mape': 17.054661530364474,
   'smape': 19.23289777226956},
  '30': {'mae': 7720.60402427584,
   'rmse': 9349.153418839429,
   'mape': 17.137543552582365,
   'smape': 19.31814001304605}}}

## 11. Forecast Horizons
Supported horizons are exactly 7, 14, and 30 days. Recursive uncertainty and missing external drivers make longer horizons less reliable.

## 12. Residual Analysis
Residual exports support analysis over time, by weekday, and for the largest under- and over-predictions.

In [7]:
residuals=pd.read_csv(report_dir/'holdout_residuals_daily_orders.csv')
display(residuals.groupby('weekday').residual.agg(['mean','std','count']))
display(residuals.reindex(residuals.residual.abs().sort_values(ascending=False).index).head())

,mean,std,count
weekday,,,
Friday,36.899129,53.995095,4
Monday,37.310633,49.149407,5
Saturday,12.340608,28.901384,4
Sunday,26.904088,41.907691,4
Thursday,48.360049,34.732947,4
Tuesday,53.500947,54.109947,5
Wednesday,48.306207,29.766948,4


,date,actual,predicted,residual,weekday
15,2018-08-07,361,239.656658,121.343342,Tuesday
14,2018-08-06,368,255.149390,112.850610,Monday
11,2018-08-03,312,217.994255,94.005745,Friday
13,2018-08-05,274,186.508505,87.491495,Sunday
24,2018-08-16,316,234.987779,81.012221,Thursday


## 13. Future Forecast
Practical uncertainty intervals use backtest residual quantiles with transparent horizon widening; they are not guaranteed confidence intervals.

In [8]:
display(generate_forecast(7))
display(generate_forecast(30).tail())

,date,predicted_orders,predicted_revenue,orders_lower_bound,orders_upper_bound,revenue_lower_bound,revenue_upper_bound
0,2018-08-22,233.697828,37675.033751,121.501293,329.421150,20283.258147,53921.845902
1,2018-08-23,231.902267,32983.990424,112.899870,333.432182,14537.226725,50216.336991
2,2018-08-24,204.072883,34229.965687,78.633343,311.094810,14785.369437,52394.453880
3,2018-08-25,182.148933,26822.821381,50.586834,294.394478,6429.156785,45873.897321
4,2018-08-26,187.072751,26794.114798,49.660620,304.309399,5493.626823,46692.314656
5,2018-08-27,247.672415,39361.322520,104.649335,369.696187,17191.071725,60072.025568
6,2018-08-28,221.665600,34262.391489,73.243535,348.295652,11255.234936,55754.903763


,date,predicted_orders,predicted_revenue,orders_lower_bound,orders_upper_bound,revenue_lower_bound,revenue_upper_bound
25,2018-09-16,196.915662,28654.587283,0.000000,391.330648,0.000000,61652.018749
26,2018-09-17,266.963247,46466.185300,35.664164,464.301931,10612.121384,79959.846588
27,2018-09-18,255.541122,41724.330237,20.865232,455.760815,5346.821641,75706.975950
28,2018-09-19,251.030843,37005.933032,13.026050,454.090674,112.405633,71470.626165
29,2018-09-20,237.672042,35222.526733,0.000000,443.532830,0.000000,70162.617401


## 14. Business Interpretation
Use forecasts as planning ranges, compare actuals daily, and escalate systematic under-prediction because it can create fulfillment and staffing shortfalls.

## 15. Limitations
The data lacks promotions, holidays, marketing, prices, macroeconomics, and current operations. Recursive error compounds, historical coverage is short, and long-term reliability is unsupported.